# LANGFENS — RQ1-TRN-03A: Dedicated Fold 2 Baseline R Training on Kaggle Tesla T4x2
Official execution notebook for Fold 2 Baseline R training (`KAGGLE_T4_CUDA_FP16_AMP`).

## Scientific Protocol & Provenance Invariants
- **Architecture:** Two-head Wav2Vec2 + Binary MDD (frozen backbone `facebook/wav2vec2-base-960h`, revision `22aad52d435eb6dbaf354bdad9b0da84ce7d6156`)
- **Initialization:** Strictly fresh pretrained initialization; zero warm-start or checkpoint transfer from Fold 1.
- **Seeding & Precision:** Seed 1337, CUDA FP16 AMP (`init_scale: 4096.0`), AdamW (`lr: 1e-3`, `weight_decay: 0.01`).
- **Batching & Step Dynamics:** `batch_size: 2`, `gradient_accumulation_steps: 2` (effective batch size = 4 utterances per update).
- **Decoupled Horizon & Cap:**
  - Scheduler Horizon: 25 epochs (~15,000 updates, 100 warmup updates) preserving Fold 1 Baseline R cosine decay rate.
  - Execution Cap: 10 completed epochs (max 6,000 updates), strictly bounding Kaggle GPU quota usage.
- **Dual Checkpoint Accounting:**
  - Fixed-Budget Primary Endpoint: Strictly captured and locked at Epoch 6 / 3,600 updates (`fixed_budget_checkpoint.pt`).
  - Validation-Selected Endpoint: Captured at peak validation AP prior to early stopping (`best.pt`, patience 5).
- **Data & Test Isolation:**
  - Dataset: Authorized Fold 2 Train+Val package (`minkhoaa/langfens-rq1-fold2-staging`).
  - Sealed Test Exclusion: Active test speakers (`BWC`, `HKK`, `HQTV`, `MBMPS`, `SKA`, `SVBI`) are 100% physically absent.
- **Credential & Durability Hygiene:**
  - Authentication: strictly via Kaggle Secrets (`HF_TOKEN`) or environment variables; zero token file fallbacks.
  - Remote Mirroring: Per-epoch atomic mirror to private Hugging Face repo `minkhoaa/langfens-rq1-cross-fold-checkpoints` with SHA-256 readback.
- **Mandatory Preflight Gate:** Runs preflight first and fails closed on any error before training begins.
- **Reviewed Training Source Commit:** `407cf04e83704d9e83953e9309e3c7dca8d7154e` (pinned training codebase).


In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys

# Checked execution of nvidia-smi
subprocess.run(["nvidia-smi"], check=True)

import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"[GPU {i}]: {torch.cuda.get_device_name(i)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), (
    f"FAIL-CLOSED: Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
)

# Verify required core dependencies
import yaml, transformers, huggingface_hub
print(f"[ENV] PyTorch version: {torch.__version__}")
print(f"[ENV] Transformers version: {transformers.__version__}")
print(f"[ENV] Hugging Face Hub version: {huggingface_hub.__version__}")


In [ ]:
# Phase 2: Credentials & Secrets Resolution (Strict Kaggle Secrets Path)
# In accordance with AGENTS.md, tokens are never read from disk files or passed via CLI.
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and token.strip():
        os.environ["HF_TOKEN"] = token.strip()
        print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[AUTH NOTE] Kaggle Secrets UserSecretsClient query: {e}")

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError(
        "FAIL-CLOSED: HF_TOKEN is required in Kaggle Secrets (or environment) for private remote mirroring. "
        "Attach HF_TOKEN under Add-ons -> Secrets in the Kaggle notebook."
    )
print("[AUTH] Verified HF_TOKEN presence in environment for private remote mirroring.")


In [ ]:
# Phase 3: Clone, Checkout Reviewed Training Commit & Verify Tree
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository into /kaggle/working/repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)],
        check=True
    )

os.chdir(str(repo_dir))
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)

# Pinned immutable training source commit (closing Step 04C launch-gate corrections)
PINNED_COMMIT_SHA = "407cf04e83704d9e83953e9309e3c7dca8d7154e"
print(f"[GIT] Checking out pinned training commit: {PINNED_COMMIT_SHA}...")
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT_SHA], cwd=repo_dir, check=True)

actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip().lower()
print(f"[GIT] Verified Training Commit HEAD: {actual_head}")
assert actual_head == PINNED_COMMIT_SHA.lower(), (
    f"FAIL-CLOSED: Commit SHA mismatch: actual {actual_head} != expected {PINNED_COMMIT_SHA}"
)

status = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_dir, text=True).strip()
assert len(status) == 0, f"Dirty tree: {status}"
print("[GIT] Repository verified clean at pinned training commit.")

sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))


In [ ]:
# Phase 4: Mandatory Preflight Check (Fail-Closed Gate)
# Training entrypoint executes preflight checks first; fails closed immediately on any error.
import subprocess, sys
from pathlib import Path

staging_input = Path("/kaggle/input/langfens-rq1-fold2-staging")
print(f"[PREFLIGHT-GATE] Auditing dataset input directory: {staging_input}")
if staging_input.is_dir():
    print(f"[PREFLIGHT-GATE] Directory contents: {[p.name for p in staging_input.iterdir()]}")
else:
    print(f"[PREFLIGHT-GATE WARNING] {staging_input} does not exist yet (requires dataset attachment).")

preflight_cmd = [
    sys.executable,
    "deploy/kaggle/kaggle_entrypoint.py",
    "--mode", "verify_fold2",
    "--commit-sha", PINNED_COMMIT_SHA,
    "--kaggle-input-dir", str(staging_input),
    "--remote-mirror-backend", "huggingface",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
]
print("[PREFLIGHT-GATE] Executing mandatory preflight verification:")
print("  " + " ".join(preflight_cmd))
subprocess.run(preflight_cmd, check=True)
print("=" * 80)
print("[PREFLIGHT] Preflight verification PASSED successfully. Proceeding to official training...")
print("=" * 80)


In [ ]:
# Phase 5: Official Training Execution (train_fold2)
# Preflight verified; launching official Fold 2 Baseline R training.
print("=" * 80)
print("[LAUNCH] Starting Official Fold 2 Baseline R Training on Tesla T4...")
print(f"[LAUNCH] Pinned Commit SHA: {PINNED_COMMIT_SHA}")
print("=" * 80)

train_cmd = [
    sys.executable,
    "deploy/kaggle/kaggle_entrypoint.py",
    "--mode", "train_fold2",
    "--commit-sha", PINNED_COMMIT_SHA,
    "--kaggle-input-dir", str(staging_input),
    "--remote-mirror-backend", "huggingface",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
]
print("[LAUNCH] Executing checked command:")
print("  " + " ".join(train_cmd))
subprocess.run(train_cmd, check=True)
print("[LAUNCH] Fold 2 Baseline R training subprocess completed successfully.")


In [ ]:
# Phase 6: Post-Training Endpoint Classification
# Enforces strict deliverable completion invariants:
# 1. fixed_budget_checkpoint.pt with exactly 3,600 updates at Epoch 6.
# 2. best.pt, last.pt, and history.json.
# 3. Correct fold index and manifest SHA.
# 4. Pinned configuration parity.
from audit_scripts.kaggle_fold_train_exec import classify_fold_training_endpoint
res = classify_fold_training_endpoint(
    "/kaggle/working/checkpoints/fold2/baseline_r_fold2",
    fold_idx=2,
    expected_commit=PINNED_COMMIT_SHA
)
print("=" * 80)
print("[FINAL ENDPOINT STATUS]:", res["status"])
print("[ENDPOINT DETAILS]:", res)
print("=" * 80)
assert res["status"] == "COMPLETED_3600_UPDATES", f"Training endpoint verification failed: {res}"
